In [ ]:
"""
Setup global paths for the project from environment variables.
"""
import os

# local
#CV_PATH_SPAIR = "./data/SPair71K"
# tfpool
CV_PATH_SPAIR = "/project/dl2026s/lmbcvtst/data/SPair71K"
os.environ["HF_HOME"] = "/project/dl2026s/lmbcvtst/data/hf_cache"


OUTPUT_DIR = "./results"
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [ ]:
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
from torchvision import transforms
from PIL import Image
import numpy as np
import torch
import glob
import json
import os

def read_img(path):
    img = np.array(Image.open(path).convert('RGB'))

    return torch.tensor(img.transpose(2, 0, 1).astype(np.float32))

class Resize(object):
    """Resize and center crop images to a fixed size."""
    def __init__(self, size=256, image_keys=['src_img']):
        self.size = size
        self.image_keys = image_keys
        self.resize_fn = transforms.Compose([
            transforms.Resize(self.size),
            transforms.CenterCrop(self.size),
        ])
    
    def __call__(self, sample):
        for key in self.image_keys:
            if key in sample:
                img_tensor = sample[key]
                # Convert tensor to PIL, apply transforms, convert back
                pil_img = transforms.ToPILImage()(img_tensor)
                resized_img = self.resize_fn(pil_img)
                sample[key] = transforms.ToTensor()(resized_img) * 255.0
        return sample


class PascalDataset(Dataset):
    """Single-image dataset from SPair71K with duplicate source filtering."""
    
    def __init__(self, pair_ann_path=None, layout_path=None, image_path=None, dataset_size='large', 
                 pck_alpha=0.1, datatype='test'):
        
        pair_ann_path = CV_PATH_SPAIR + '/PairAnnotation'
        layout_path = CV_PATH_SPAIR + '/Layout'
        image_path = CV_PATH_SPAIR + '/JPEGImages'
        
        self.datatype = datatype
        self.pck_alpha = pck_alpha
        self.pair_ann_path = pair_ann_path
        self.image_path = image_path
        self.categories = list(map(lambda x: os.path.basename(x), glob.glob('%s/*' % image_path)))
        self.categories.sort()
        self.resize = Resize(size=256, image_keys=['src_img'])
        self.transform = torch.nn.Identity()
        
        # Load annotation files
        ann_files = open(os.path.join(layout_path, dataset_size, datatype + '.txt'), "r").read().split('\n')
        ann_files = ann_files[:len(ann_files) - 1]
        
        # Filter duplicates based on source image names
        seen_sources = set()
        self.ann_files = []
        
        for ann_file in ann_files:
            ann_json = ann_file + '.json'
            ann_path = os.path.join(pair_ann_path, datatype, ann_json)
            
            with open(ann_path) as f:
                annotation = json.load(f)
                src_imname = annotation['src_imname']
                
                # Only add if source image not seen before
                if src_imname not in seen_sources:
                    seen_sources.add(src_imname)
                    self.ann_files.append(ann_file)
        
        print(f"PascalDataset: Filtered from {len(ann_files)} to {len(self.ann_files)} samples (removed {len(ann_files) - len(self.ann_files)} duplicates)")
    
    def __len__(self):
        return len(self.ann_files)
    
    def __getitem__(self, idx):
        # get pre-processed source images only
        ann_file = self.ann_files[idx] + '.json'
        with open(os.path.join(self.pair_ann_path, self.datatype, ann_file)) as f:
            annotation = json.load(f)
        
        category = annotation['category']
        src_img = read_img(os.path.join(self.image_path, category, annotation['src_imname']))
        
        src_bbox = annotation['src_bndbox']
        pck_threshold = max(src_bbox[2] - src_bbox[0], src_bbox[3] - src_bbox[1]) * self.pck_alpha
        
        sample = {
            'pair_id': annotation['pair_id'],
            'src_imname': annotation['src_imname'],
            'src_bbox': annotation['src_bndbox'],
            'category': annotation['category'],
            'src_pose': annotation['src_pose'],
            'src_img': src_img,
            'src_kps': torch.tensor(annotation['src_kps']).float(),
            'pck_threshold': pck_threshold
        }
        
        if self.resize:
            sample = self.resize(sample)
        # Record imsize after resize so all samples have identical shape
        sample['src_imsize'] = list(sample['src_img'].shape)
        sample['src_img_orig'] = sample['src_img'].clone()
        if self.transform:
            sample = self.transform(sample)
        
        return sample



from torch.utils.data.dataloader import default_collate

def collate_var_len(batch):
    """Custom collate that keeps variable-length tensors as lists instead of stacking."""
    elem = batch[0]
    result = {}
    for key in elem:
        values = [d[key] for d in batch]
        if isinstance(values[0], torch.Tensor) and any(v.shape != values[0].shape for v in values):
            result[key] = values
        else:
            try:
                result[key] = default_collate(values)
            except RuntimeError:
                result[key] = values
    return result


# Test datasets
print(f"SPAIR71K path: {CV_PATH_SPAIR}")
print(f"SPAIR71K exists: {os.path.exists(CV_PATH_SPAIR)}")

if os.path.exists(CV_PATH_SPAIR):
    print("\n" + "="*60)
    print("Testing SPairDataset and PascalDataset")
    print("="*60)
    
    ssingle_test = PascalDataset(datatype='test')
    
    print(f"SSingle Test:  {len(ssingle_test):6d} unique source images")
    
    if len(ssingle_test) > 0:
        sample = ssingle_test[0]
        print(f"\nSample category: {sample['category']}")
        print(f"Source image shape: {sample['src_img'].shape}")
        #display(transforms.ToPILImage()(sample['src_img']))

In [ ]:
import torch.nn.functional as F
import open_clip
from transformers import AutoImageProcessor, AutoModel
from tqdm import tqdm
import matplotlib.pyplot as plt

# Setup device
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

@torch.no_grad()
def extract_features(inputs, model_type='dino', feature_type='image', batch_size=32):
    """
    General feature extraction function for DINO or CLIP models.
    
    Args:
        inputs: 
            - For image features: list of PIL images or torch tensors [B, C, H, W], dataset object, or dataset slice
            - For text features: list of text strings
        model_type: 'dino' or 'clip'
        feature_type: 'image' or 'text'
        batch_size: batch size for processing
    
    Returns:
        embeddings: torch tensor of shape [N, embedding_dim]
        
    Examples:
        # Extract image embeddings from DINO
        images = [Image.open(path) for path in image_paths]
        features = extract_features(images, model_type='dino', feature_type='image')
        
        # Extract text embeddings from CLIP
        texts = ['a cat', 'a dog', 'a person']
        features = extract_features(texts, model_type='clip', feature_type='text')
        
        # Extract from dataset or dataset slice
        features = extract_features(dataset, model_type='dino', feature_type='image')
        features = extract_features(dataset[:10], model_type='dino', feature_type='image')
    """
    
    # Setup model
    if model_type.lower() == 'dino':
        processor = AutoImageProcessor.from_pretrained('facebook/dinov2-base')
        backbone = AutoModel.from_pretrained('facebook/dinov2-base')
        backbone = backbone.to(DEVICE).eval()
        for param in backbone.parameters():
            param.requires_grad = False
    elif model_type.lower() == 'clip':
        clip_model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-16', pretrained='laion2b_s34b_b88k')
        clip_model = clip_model.to(DEVICE).eval()
        tokenizer = open_clip.get_tokenizer('ViT-B-16')
        for param in clip_model.parameters():
            param.requires_grad = False
    else:
        raise ValueError(f"Unknown model_type: {model_type}")
    
    all_embeddings = []
    
    # Check if input is a list of dicts (dataset slice or batch)
    if isinstance(inputs, list) and len(inputs) > 0 and isinstance(inputs[0], dict):
        # This is a dataset slice or batch of samples
        for i in tqdm(range(0, len(inputs), batch_size), desc=f"Extracting {model_type} {feature_type} features"):
            batch_samples = inputs[i:i+batch_size]
            
            if feature_type == 'image':
                batch_images = [sample['src_img'] for sample in batch_samples]
                
                # Convert to PIL if needed
                pil_images = []
                for img in batch_images:
                    if isinstance(img, Image.Image):
                        pil_images.append(img)
                    elif isinstance(img, torch.Tensor):
                        # Normalize to [0,1] float for ToPILImage regardless of original scale
                        img_norm = img.float()
                        if img_norm.max() > 1.0:
                            img_norm = img_norm / 255.0
                        pil_images.append(transforms.ToPILImage()(img_norm.clamp(0, 1)))
                    else:
                        raise ValueError(f"Unsupported image type: {type(img)}")
                
                if model_type.lower() == 'dino':
                    processed = processor(pil_images, return_tensors='pt')
                    pixel_values = processed['pixel_values'].to(DEVICE)
                    outputs = backbone(pixel_values)
                    emb = outputs.last_hidden_state[:, 0, :]  # CLS token
                else:  # clip
                    processed_images = torch.stack([preprocess(img) for img in pil_images]).to(DEVICE)
                    emb = clip_model.encode_image(processed_images)
                
                all_embeddings.append(emb.cpu())
    
    # Check if input is a dataset object (has categories attribute)
    elif hasattr(inputs, 'categories'):
        loader = DataLoader(inputs, batch_size=batch_size, shuffle=False, collate_fn=collate_var_len)
        for batch in tqdm(loader, desc=f"Extracting {model_type} {feature_type} features"):
            if feature_type == 'image':
                images = batch['src_img']
                
                # Safely convert to PIL (handle both raw [0,255] and normalized [0,1] tensors)
                def to_pil(img):
                    img_f = img.float()
                    if img_f.max() > 1.0:
                        img_f = img_f / 255.0
                    return transforms.ToPILImage()(img_f.clamp(0, 1))
                if model_type.lower() == 'dino':
                    pil_images = [to_pil(img) for img in images]
                    processed = processor(pil_images, return_tensors='pt')
                    pixel_values = processed['pixel_values'].to(DEVICE)
                    outputs = backbone(pixel_values)
                    emb = outputs.last_hidden_state[:, 0, :]  # CLS token
                else:  # clip
                    pil_images = [to_pil(img) for img in images]
                    processed_images = torch.stack([preprocess(img) for img in pil_images]).to(DEVICE)
                    emb = clip_model.encode_image(processed_images)
                
                all_embeddings.append(emb.cpu())
        
        embeddings = torch.cat(all_embeddings, dim=0)
        return F.normalize(embeddings, dim=1)
    
    # Handle list of images or texts
    elif isinstance(inputs, list):
        if feature_type == 'image':
            for i in tqdm(range(0, len(inputs), batch_size), desc=f"Extracting {model_type} image features"):
                batch_images = inputs[i:i+batch_size]
                
                # Convert to PIL if needed
                pil_images = []
                for img in batch_images:
                    if isinstance(img, Image.Image):
                        pil_images.append(img)
                    elif isinstance(img, torch.Tensor):
                        # Normalize to [0,1] float for ToPILImage regardless of original scale
                        img_norm = img.float()
                        if img_norm.max() > 1.0:
                            img_norm = img_norm / 255.0
                        pil_images.append(transforms.ToPILImage()(img_norm.clamp(0, 1)))
                    else:
                        raise ValueError(f"Unsupported image type: {type(img)}")
                
                if model_type.lower() == 'dino':
                    processed = processor(pil_images, return_tensors='pt')
                    pixel_values = processed['pixel_values'].to(DEVICE)
                    outputs = backbone(pixel_values)
                    emb = outputs.last_hidden_state[:, 0, :]  # CLS token
                else:  # clip
                    processed_images = torch.stack([preprocess(img) for img in pil_images]).to(DEVICE)
                    emb = clip_model.encode_image(processed_images)
                
                all_embeddings.append(emb.cpu())
        
        elif feature_type == 'text':
            # Process text in batches (only CLIP supports text)
            if model_type.lower() != 'clip':
                raise ValueError("Text feature extraction only supported for CLIP")
            
            for i in tqdm(range(0, len(inputs), batch_size), desc="Extracting CLIP text features"):
                batch_texts = inputs[i:i+batch_size]
                tokens = tokenizer(batch_texts).to(DEVICE)
                emb = clip_model.encode_text(tokens)
                all_embeddings.append(emb.cpu())
        
        else:
            raise ValueError(f"Unknown feature_type: {feature_type}")
    
    else:
        raise ValueError(f"Unsupported input type: {type(inputs)}")
    
    embeddings = torch.cat(all_embeddings, dim=0)
    return F.normalize(embeddings, dim=1)  # Return normalized embeddings


## 🎯 Exercises 1 & 2: Unified k-Nearest Neighbors Classification

Implement a single kNN classifier that works with both DINO and OpenCLIP backbones.

### The KNNClassifier

Create a `KNNClassifier` class that:
1. Accepts a `backbone_type` parameter ('dino' or 'clip')
2. Extracts embeddings from either DINO or OpenCLIP
3. Stores embeddings and category labels
4. Performs kNN prediction with majority voting
5. Computes classification accuracy

### Compare embeddings:
- DINO: self-supervised, trained on unlabeled images (768-dim)
- OpenCLIP: trained on image-text pairs (512-dim)
- Both should extract useful features from different training paradigms

In [ ]:
class KNNClassifier:
    """Unified k-Nearest Neighbors classifier for DINO and CLIP backbones."""

    def __init__(self, backbone_type='dino', k=5):
        """
        Args:
            backbone_type: 'dino' or 'clip'
            k: Number of nearest neighbors
        """
        self.backbone_type = backbone_type.lower()
        self.k = k

        if self.backbone_type not in ['dino', 'clip']:
            raise ValueError(f"Unknown backbone_type: {backbone_type}")

        self.train_embeddings = None
        self.train_labels = None

    def fit(self, train_dataset):
        """Fit classifier on training data."""
        print(f"Extracting {self.backbone_type.upper()} embeddings for training...")
        # TODO 1: Extract image embeddings for train_dataset using extract_features().
        #         Use model_type=self.backbone_type and feature_type='image'.
        #         Store the result in self.train_embeddings.
        self.train_embeddings = extract_features(train_dataset, self.backbone_type, feature_type='image')
        # TODO 2: Build self.train_labels as a list of integer class indices.
        #         Iterate over train_dataset with DataLoader(batch_size=32, shuffle=False,
        #         collate_fn=collate_var_len) and map each batch['category'] string to its
        #         index via train_dataset.categories.index(cat).
        _loader = DataLoader(train_dataset, batch_size=32, shuffle=False, collate_fn=collate_var_len)
        self.train_labels = []
        for _batch in _loader:
            self.train_labels.extend([train_dataset.categories.index(cat) for cat in _batch['category']])

    @torch.no_grad()
    def predict(self, test_dataset):
        """Predict labels for test set using kNN."""
        print(f"Extracting {self.backbone_type.upper()} embeddings for testing...")
        # TODO 3: Extract image embeddings for test_dataset and build test_labels
        #         using the same approach as fit().
        self.test_embeddings = extract_features(test_dataset, self.backbone_type, feature_type='image')
        _loader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=collate_var_len)
        self.test_labels = []
        for _batch in _loader:
            self.test_labels.extend([test_dataset.categories.index(cat) for cat in _batch['category']])
        # TODO 4: For each test embedding find the k nearest neighbours:
        #           - DINO: compute L2 distance to every training embedding; take k smallest.
        #           - CLIP: compute L2 distance to every training embedding; take k smallest.
        #         Predict via majority vote over the k neighbour labels.
        #         Collect all predictions in a list called `predictions`.
        #
        # The method must return (predictions, accuracy, test_labels).
        test_labels_t = torch.tensor(self.test_labels)
        train_labels_t = torch.tensor(self.train_labels)
        distances = torch.cdist(self.test_embeddings, self.train_embeddings, p=2)        
        _, knn_indices = torch.topk(distances, self.k, dim=1, largest=False)
        
        knn_labels = train_labels_t[knn_indices]
        predictions = []
        for i in range(knn_labels.shape[0]):
            mode_result = torch.mode(knn_labels[i])
            predictions.append(mode_result.values.item())
            predictions = torch.tensor(predictions)
        accuracy = (predictions == test_labels_t).float().mean().item()
        
        return predictions.tolist(), accuracy, self.test_labels           
        


# Test unified kNN classifiers
if os.path.exists(CV_PATH_SPAIR):
    print("\n" + "="*60)
    print("Exercises 1 & 2: Unified kNN Classifiers")
    print("="*60)

    train_data = PascalDataset(datatype='trn')
    test_data = PascalDataset(datatype='test')

    # DINO classifier
    print("\n--- DINO Backbone ---")
    dino_clf = KNNClassifier(backbone_type='dino', k=5)
    print(f"Training on {len(train_data)} images...")
    dino_clf.fit(train_data)
    print(f"Testing on {len(test_data)} images...")
    dino_pred, dino_acc, _ = dino_clf.predict(test_data)
    print(f"\u2713 DINO kNN Accuracy: {dino_acc:.4f}")

    # CLIP classifier
    print("\n--- OpenCLIP Backbone ---")
    clip_clf = KNNClassifier(backbone_type='clip', k=5)
    print(f"Training on {len(train_data)} images...")
    clip_clf.fit(train_data)
    print(f"Testing on {len(test_data)} images...")
    clip_pred, clip_acc, _ = clip_clf.predict(test_data)
    print(f"\u2713 OpenCLIP kNN Accuracy: {clip_acc:.4f}")

    # Compare
    print("\n" + "="*60)
    print("Model Comparison")
    print("="*60)
    print(f"DINO:      {dino_acc:.4f}")
    print(f"OpenCLIP:  {clip_acc:.4f}")

    # Save kNN accuracies
    with open(os.path.join(OUTPUT_DIR, "accuracies.json"), "w") as _f:
        json.dump({"dino_knn_k5": float(dino_acc), "clip_knn_k5": float(clip_acc)}, _f, indent=2)
    print(f"Saved kNN accuracies to {OUTPUT_DIR}/accuracies.json")


In [ ]:
if os.path.exists(CV_PATH_SPAIR):
    print("\n" + "="*60)
    print("kNN Accuracy Sweep: k = 1 to 10")
    print("="*60)

    # Extract test embeddings once so we don't reload models 10x
    print("Extracting DINO test embeddings...")
    dino_test_embs = extract_features(test_data, model_type='dino', feature_type='image')
    print("Extracting CLIP test embeddings...")
    clip_test_embs = extract_features(test_data, model_type='clip', feature_type='image')

    # Test labels
    _loader = DataLoader(test_data, batch_size=32, shuffle=False, collate_fn=collate_var_len)
    test_labels_list = []
    for _batch in _loader:
        test_labels_list.extend([test_data.categories.index(c) for c in _batch['category']])
    train_labels_t = torch.tensor(dino_clf.train_labels)

    # Precompute full distance/similarity matrices  [N_test x N_train]
    dino_dists = torch.cdist(dino_test_embs, dino_clf.train_embeddings)
    clip_dists = torch.cdist(clip_test_embs, clip_clf.train_embeddings)

    def majority_vote(nn_indices, labels_t):
        preds = []
        for row in nn_indices:
            nbrs = labels_t[row].tolist()
            preds.append(max(set(nbrs), key=nbrs.count))
        return preds

    k_values = list(range(1, 11))
    dino_accs, clip_accs = [], []

    for k in k_values:
        _, dino_nn = torch.topk(dino_dists, k, dim=1, largest=False)
        dino_preds = majority_vote(dino_nn, train_labels_t)
        dino_accs.append(float(np.mean([p == t for p, t in zip(dino_preds, test_labels_list)])))

        _, clip_nn = torch.topk(clip_dists, k, dim=1, largest=False)
        clip_preds = majority_vote(clip_nn, train_labels_t)
        clip_accs.append(float(np.mean([p == t for p, t in zip(clip_preds, test_labels_list)])))

        print(f"  k={k:2d}  DINO={dino_accs[-1]:.4f}  CLIP={clip_accs[-1]:.4f}")

    # Plot
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(k_values, dino_accs, 'o-', label='DINO (self-supervised)',
            color='steelblue', linewidth=2, markersize=7)
    ax.plot(k_values, clip_accs, 's--', label='OpenCLIP (image-text)',
            color='darkorange', linewidth=2, markersize=7)

    ax.set_xlabel('k  (number of nearest neighbours)', fontsize=12)
    ax.set_ylabel('Top-1 Accuracy', fontsize=12)
    ax.set_title('kNN Classification Accuracy vs k\n(SPair-71K test split, 18 categories)',
                 fontsize=13)
    ax.set_xticks(k_values)
    ax.set_ylim(0, 1.0)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'knn_accuracy_vs_k.png'), dpi=150, bbox_inches='tight')
    plt.show()

    best_dino = k_values[int(np.argmax(dino_accs))]
    best_clip = k_values[int(np.argmax(clip_accs))]
    print(f"\nBest DINO:     k={best_dino}, acc={max(dino_accs):.4f}")
    print(f"Best OpenCLIP: k={best_clip}, acc={max(clip_accs):.4f}")

    # Save k-sweep accuracy data
    sweep_data = {"k_values": k_values, "dino_accs": dino_accs, "clip_accs": clip_accs}
    with open(os.path.join(OUTPUT_DIR, "knn_sweep.json"), "w") as _f:
        json.dump(sweep_data, _f, indent=2)
    print(f"Saved k-sweep data to {OUTPUT_DIR}/knn_sweep.json")


## 🎯 Exercise 3: Zero-Shot Classification with OpenCLIP

OpenCLIP was trained with both vision and text encoders. Now you'll use the text encoder to perform zero-shot classification!

### Zero-Shot Classifier

Create an `OpenCLIPZeroShotClassifier` class that:
1. Uses OpenCLIP's text encoder to embed category names
2. Computes similarity between image and text embeddings
3. Performs classification without any training on the specific categories
4. Compare with the supervised kNN results

### Key insight:
This shows the power of multimodal training—the model can classify images of categories it was never explicitly trained on, just by knowing their names!

In [ ]:
class OpenCLIPZeroShotClassifier:
    """Zero-shot classification using OpenCLIP text and vision encoders."""

    def __init__(self):
        self.text_embeddings = None
        self.categories = None
        self.category_to_idx = None

    def fit(self, dataset):
        """Prepare text embeddings for categories."""
        self.categories = sorted(list(set(dataset.categories)))
        self.category_to_idx = {cat: idx for idx, cat in enumerate(self.categories)}
        # TODO 5: Create a list of text prompts, one per category.
        #         Recommended format: "a photo of <category>".
        text_prompts = [f"a photo of {cat}" for cat in self.categories]
        # TODO 6: Extract text embeddings via extract_features() with
        #         model_type='clip' and feature_type='text'.
        #         Store in self.text_embeddings.
        self.text_embeddings = extract_features(text_prompts, model_type='clip', feature_type='text')

    def predict(self, test_dataset):
        """Predict labels using zero-shot classification."""
        # TODO 7: Extract CLIP image embeddings for test_dataset.
        image_embeddings = extract_features(test_dataset, model_type='clip', feature_type='image')

        # TODO 8: Compute cosine similarity between each image embedding and all
        #         text embeddings (self.text_embeddings). Predict the category index
        #         with the highest similarity.
        image_embeddings_norm = F.normalize(image_embeddings, dim=1, p=2)
        text_embeddings_norm = F.normalize(self.text_embeddings, dim=1, p=2)
        similarities = torch.mm(image_embeddings_norm, text_embeddings_norm.t())
        predictions = torch.argmax(similarities, dim=1)

        # TODO 9: Build true_labels with a DataLoader and self.category_to_idx,
        #         compute accuracy, and return (predictions, accuracy, true_labels).
        _loader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=collate_var_len)
        true_labels = []
        for _batch in _loader:
            true_labels.extend([self.category_to_idx[cat] for cat in _batch['category']])
        true_labels = torch.tensor(true_labels)
        
        # Compute accuracy
        accuracy = (predictions == true_labels).float().mean().item()
        
        return predictions.tolist(), accuracy, true_labels.tolist()

# Test zero-shot classifier
if os.path.exists(CV_PATH_SPAIR):
    print("\n" + "="*60)
    print("Testing Zero-Shot Classifier")
    print("="*60)

    zeroshot_classifier = OpenCLIPZeroShotClassifier()
    print("Preparing text embeddings...")
    zeroshot_classifier.fit(test_data)

    print("Making zero-shot predictions...")
    predictions_zeroshot, accuracy_zeroshot, _ = zeroshot_classifier.predict(test_data)
    print(f"Zero-Shot Accuracy: {accuracy_zeroshot:.4f}")

    # Compare all three approaches
    print("\n" + "="*60)
    print("Full Model Comparison")
    print("="*60)
    print(f"DINO kNN:        {dino_acc:.4f}")
    print(f"OpenCLIP kNN:    {clip_acc:.4f}")
    print(f"Zero-Shot:       {accuracy_zeroshot:.4f}")

    # Append zero-shot accuracy to results file
    _acc_path = os.path.join(OUTPUT_DIR, "accuracies.json")
    if os.path.exists(_acc_path):
        with open(_acc_path) as _f:
            _accs = json.load(_f)
    else:
        _accs = {}
    _accs["zeroshot"] = float(accuracy_zeroshot)
    with open(_acc_path, "w") as _f:
        json.dump(_accs, _f, indent=2)
    print(f"Saved zero-shot accuracy to {_acc_path}")


## 🎯 Exercise 4: Text-to-Image Retrieval

The final exercise demonstrates text-to-image retrieval: given a text query, find the most similar images.

### Text-to-Image Retriever

Create a `TextToImageRetriever` class that:
1. Builds an index of all image embeddings
2. Takes free-form text queries (e.g., "fluffy cat", "red car")
3. Encodes the query to embedding space
4. Returns top-k most similar images by similarity score
5. Visualizes results for qualitative evaluation

### Applications:
- Image search engines
- Content recommendation
- Dataset exploration
- Understanding what models learn

In [ ]:
class TextToImageRetriever:
    """Text-to-image retrieval using OpenCLIP embeddings."""

    def __init__(self, model_name='ViT-B-16', pretrained='laion2b_s34b_b88k'):
        self.model_name = model_name
        self.pretrained = pretrained

        clip_model, _, _ = open_clip.create_model_and_transforms(model_name, pretrained=pretrained)
        self.clip_model = clip_model.to(DEVICE).eval()
        self.tokenizer = open_clip.get_tokenizer(model_name)

        for param in self.clip_model.parameters():
            param.requires_grad = False

        self.image_embeddings = None
        self.image_categories = None

    def build_index(self, dataset):
        """Build index of all image embeddings."""
        # TODO 10: Extract CLIP image embeddings for all images in dataset using
        #          extract_features(). Store in self.image_embeddings.
        print("Extracting image embeddings for retrieval...")
        self.image_embeddings = extract_features(dataset, model_type='clip', feature_type='image')
        
        # TODO 11: Build self.image_categories as a list of category strings.
        #          Use DataLoader(batch_size=32, collate_fn=collate_var_len) and
        #          extend from batch['category'].
        _loader = DataLoader(dataset, batch_size=32, shuffle=False, collate_fn=collate_var_len)
        self.image_categories = []
        for _batch in _loader:
            self.image_categories.extend(_batch['category'])
        
        print(f"Built index with {len(self.image_embeddings)} images")
        

    @torch.no_grad()
    def search(self, query, top_k=5):
        """
        Search for images matching text query.

        Args:
            query: Text description (e.g., "a red car", "fluffy animal")
            top_k: Number of results to return

        Returns:
            List of dicts with score, category, index
        """
        # TODO 12: Tokenize `query` with self.tokenizer([query]), move to DEVICE,
        #          encode with self.clip_model.encode_text(), and L2-normalise with
        #          F.normalize(dim=1).
        tokens = self.tokenizer([query]).to(DEVICE)
        with torch.no_grad():
            query_embedding = self.clip_model.encode_text(tokens)  # [1, dim]
        query_embedding = F.normalize(query_embedding, dim=1, p=2)

        # TODO 13: Compute L2 distance between the query embedding and every entry in
        #          self.image_embeddings. Return the top_k closest results as a list
        #          of dicts: [{'score': float, 'category': str, 'index': int}, ...]
        image_embeddings_norm = F.normalize(self.image_embeddings, dim=1, p=2)
        similarities = torch.mm(query_embedding, image_embeddings_norm.t()).squeeze(0)
        top_scores, top_indices = torch.topk(similarities, top_k, largest=True)
        
        results = []
        for score, idx in zip(top_scores, top_indices):
            results.append({
                'score': score.item(),
                'category': self.image_categories[idx],
                'index': idx.item()
            })
        
        return results
       

    def visualize_results(self, query, dataset, top_k=5, save_dir=None):
        """Visualize top-k search results."""
        results = self.search(query, top_k=top_k)

        print(f"\nQuery: '{query}'")
        print(f"Top-{top_k} Results:")
        print("-" * 60)

        fig, axes = plt.subplots(1, top_k, figsize=(15, 3))
        if top_k == 1:
            axes = [axes]

        for i, (ax, result) in enumerate(zip(axes, results)):
            idx = result['index']
            sample = dataset[idx]
            ax.imshow(transforms.ToPILImage()(sample['src_img_orig']))
            ax.set_title(f"{result['category']}\n({result['score']:.3f})", fontsize=10)
            ax.axis('off')
            print(f"  {i+1}. {result['category']:15s} | Distance:   {result['score']:.4f}")

        plt.tight_layout()
        if save_dir is not None:
            _safe_q = query.replace(" ", "_").replace("/", "_")
            plt.savefig(os.path.join(save_dir, f"retrieval_{_safe_q}.png"), dpi=150, bbox_inches="tight")
            print(f"Saved retrieval plot to {save_dir}/retrieval_{_safe_q}.png")
        plt.show()


# Test text-to-image retrieval
if os.path.exists(CV_PATH_SPAIR):
    print("\n" + "="*60)
    print("Testing Text-to-Image Retrieval")
    print("="*60)

    retriever = TextToImageRetriever()
    print("Building image index...")
    retriever.build_index(test_data)

    test_queries = [
        "chair inside",
        "chair outside",
    ]

    for query in test_queries[:2]:
        retriever.visualize_results(query, test_data, top_k=5, save_dir=OUTPUT_DIR)
